In [1]:
import sys
from pathlib import Path

# Bootstrap — find src/ dynamically, works on any machine
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / '.env').exists()) / 'src'))

from config import *
from download_log import load_log, update_entry, print_entry, print_stale_sources
from datetime import datetime
import pandas as pd
import os
import requests

log = load_log()
print(f"Log loaded. Rows: {len(log)}")
print(f"ACLED_EMAIL configured: {bool(ACLED_EMAIL)}")

Log loaded. Rows: 8
ACLED_EMAIL configured: True


## ACLED Pipeline

**Source:** Armed Conflict Location and Event Data Project
**Access:** Cookie-based authentication using ACLED credentials stored in `.env`
**Download instructions:** See `docs/instructions_data_maintenance.md` — ACLED section

### Framework usage
| Indicator | Concept | Role |
|-----------|---------|------|
| Battle events count | Political stability | Primary tier 1 |
| Fatalities | Political stability | Primary tier 1 |
| Protests/riots count | Political stability | Primary tier 1 |
| Civilian violence count | Political stability | Primary tier 1 |

In [2]:
import requests
from datetime import datetime

ACLED_LOGIN_URL = "https://acleddata.com/user/login?_format=json"
ACLED_API_URL = "https://acleddata.com/api/acled/read"

def get_acled_session():
    """Authenticate with ACLED and return a session with cookies."""
    session = requests.Session()
    response = session.post(
        ACLED_LOGIN_URL,
        json={"name": ACLED_EMAIL, "pass": ACLED_PASSWORD},
        timeout=30
    )
    if response.status_code == 200:
        data = response.json()
        if 'csrf_token' in data:
            print(f"Authenticated as: {data.get('current_user', {}).get('name', 'unknown')}")
            return session
        else:
            print(f"Authentication failed: {data}")
            return None
    else:
        print(f"Login request failed: {response.status_code}")
        return None

session = get_acled_session()

Authenticated as: mjboulanger@hotmail.com


In [3]:
import pandas as pd
from datetime import datetime

# ACLED event types relevant to framework
# battles, explosions/remote violence, violence against civilians, protests, riots
EVENT_TYPES = [
    'Battles',
    'Explosions/Remote violence',
    'Violence against civilians',
    'Protests',
    'Riots'
]

def fetch_acled_year(session, year, page_size=5000):
    """Fetch all ACLED events for a given year."""
    all_rows = []
    page = 1
    while True:
        params = {
            'year': year,
            'fields': 'country|iso|year|event_type|fatalities',
            'limit': page_size,
            'page': page,
            'format': 'json',
        }
        response = session.get(ACLED_API_URL, params=params, timeout=60)
        if response.status_code != 200:
            print(f"  Error {response.status_code} on page {page}")
            break
        data = response.json()
        rows = data.get('data', [])
        if not rows:
            break
        all_rows.extend(rows)
        count = data.get('count', 0)
        print(f"  Year {year}, page {page}: {len(rows)} rows (total so far: {len(all_rows)} of {count})")
        if len(all_rows) >= int(count):
            break
        page += 1
    return all_rows

# Test with one recent year first
print("Testing fetch for 2023...")
test_rows = fetch_acled_year(session, 2023)
print(f"Total rows fetched: {len(test_rows)}")
if test_rows:
    df_test = pd.DataFrame(test_rows)
    print(f"Columns: {list(df_test.columns)}")
    print(df_test.head(3))

Testing fetch for 2023...
  Error 403 on page 1
Total rows fetched: 0
